# Example 1: Binary to BCD Converter (Combinational Logic)

## Overview
This notebook demonstrates using AutoChip to generate a binary-to-BCD converter.

**Module Specification:**
- Name: `binary_to_bcd_converter`
- Input: 5-bit binary number (0-31)
- Output: 8-bit BCD (tens digit in upper 4 bits, ones digit in lower 4 bits)
- Type: Combinational logic

**Function:**
- `bcd_output[3:0] = binary_input % 10` (ones digit)
- `bcd_output[7:4] = binary_input / 10` (tens digit)

In [1]:
#@title Setting up the notebook

### Installing dependencies
!pip install openai tiktoken

!apt-get update
!apt-get install -y iverilog

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,742 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,299 kB]
Get:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,901 kB

In [2]:
#@title Utility functions

import sys
import os
import openai
import tiktoken
from abc import ABC, abstractmethod
import re
import getopt
import json
import subprocess
from time import time

class ModelInterface(ABC):
    @abstractmethod
    def generate_conversation(self, messages, temperature=0.0, max_tokens=512, num_candidates=1):
        pass

class OpenAIModel(ModelInterface):
    def __init__(self, model_id="gpt-4o-mini"):
        self.model_id = model_id
        self.client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])

    def generate_conversation(self, messages, temperature=0.0, max_tokens=512, num_candidates=1):
        responses = []
        for _ in range(num_candidates):
            try:
                completion = self.client.chat.completions.create(
                    model=self.model_id,
                    messages=messages,
                    temperature=temperature,
                    max_tokens=max_tokens
                )
                responses.append(completion.choices[0].message.content)
            except Exception as e:
                print(f"Error generating response: {e}")
                responses.append("")
        return responses

def extract_verilog_code(text):
    """Extract Verilog module code from text, handling markdown code blocks."""
    # Try to find code in markdown blocks first
    code_block_match = re.search(r'```(?:verilog)?\s*\n(.*?)```', text, re.DOTALL)
    if code_block_match:
        text = code_block_match.group(1)

    # Extract module...endmodule
    module_match = re.search(r'(?s)\bmodule\b.*?\bendmodule\b', text)
    if module_match:
        return module_match.group(0)
    return None

def run_iverilog_test(design_file, testbench_file, work_dir="."):
    """Compile and run Verilog with iverilog."""
    try:
        # Compile
        compile_cmd = f"cd {work_dir} && iverilog -g2012 -o sim.vvp {design_file} {testbench_file}"
        result = subprocess.run(compile_cmd, shell=True, capture_output=True, text=True, timeout=30)

        if result.returncode != 0:
            return False, f"Compilation error:\n{result.stderr}"

        # Simulate
        sim_cmd = f"cd {work_dir} && vvp sim.vvp"
        result = subprocess.run(sim_cmd, shell=True, capture_output=True, text=True, timeout=30)

        output = result.stdout + result.stderr

        # Check for success
        if "All test cases passed" in output:
            return True, output
        else:
            return False, output

    except subprocess.TimeoutExpired:
        return False, "Simulation timeout"
    except Exception as e:
        return False, f"Error: {str(e)}"

def verilog_loop(design_prompt, module_name, testbench_file, max_iterations, model, work_dir=".", num_candidates=5):
    """AutoChip main loop with trajectory tracking."""

    trajectory = []
    conversation_history = [{"role": "user", "content": design_prompt}]

    for iteration in range(max_iterations):
        print(f"\n{'='*60}")
        print(f"ITERATION {iteration + 1}/{max_iterations}")
        print(f"{'='*60}\n")

        # Generate candidates
        responses = model.generate_conversation(
            conversation_history,
            temperature=0.7 if num_candidates > 1 else 0.0,
            max_tokens=1024,
            num_candidates=num_candidates
        )

        best_code = None
        best_result = None

        # Test each candidate
        for idx, response in enumerate(responses):
            print(f"\nTesting candidate {idx + 1}/{num_candidates}...")

            verilog_code = extract_verilog_code(response)
            if not verilog_code:
                print("  ❌ No valid Verilog code found")
                continue

            # Save design
            design_file = f"{work_dir}/design_{iteration}_{idx}.v"
            with open(design_file, 'w') as f:
                f.write(verilog_code)

            # Test
            success, output = run_iverilog_test(
                os.path.basename(design_file),
                testbench_file,
                work_dir
            )

            if success:
                print("  ✅ TEST PASSED!")
                trajectory.append({
                    "iteration": iteration + 1,
                    "candidate": idx + 1,
                    "status": "success",
                    "code": verilog_code,
                    "output": output
                })
                return True, verilog_code, trajectory
            else:
                print(f"  ❌ Test failed")
                if best_code is None:
                    best_code = verilog_code
                    best_result = output

        # All candidates failed, add feedback
        trajectory.append({
            "iteration": iteration + 1,
            "status": "failed",
            "error": best_result
        })

        feedback = f"\n\nThe previous design failed with this error:\n{best_result}\n\nPlease fix the design."
        conversation_history.append({"role": "assistant", "content": responses[0]})
        conversation_history.append({"role": "user", "content": feedback})

    return False, None, trajectory

print("✅ Utility functions loaded successfully")

✅ Utility functions loaded successfully


## Setting the API Key

**Important:** Insert your OpenAI API key in the cell below.

In [ ]:
### OpenAI API KEY
# INSERT YOUR API KEY HERE
os.environ["OPENAI_API_KEY"] = ""

# Verify it's set
if os.environ.get("OPENAI_API_KEY") == "":
    print("⚠️  Please replace 'your-api-key-here' with your actual OpenAI API key")
else:
    print("✅ API key is set")

⚠️  Please replace 'your-api-key-here' with your actual OpenAI API key


## Setting up files and configuration

We'll download the testbench from the ChipChat example and create our configuration.

In [4]:
#@title Setting up files

# Create working directory
!mkdir -p binary_to_bcd

# Download testbench
!cd binary_to_bcd && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/binary_to_bcd_tb.v

print("\n✅ Files downloaded successfully")
!ls -lh binary_to_bcd/

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1078  100  1078    0     0   4286      0 --:--:-- --:--:-- --:--:--  4294

✅ Files downloaded successfully
total 4.0K
-rw-r--r-- 1 root root 1.1K Feb 15 21:29 binary_to_bcd_tb.v


## Testbench Code (Commented)

The testbench validates our design by testing all 32 possible inputs (0-31) and checking that each produces the correct BCD output.

In [5]:
#@title View and Explain Testbench

# Display the testbench code
print("=== TESTBENCH CODE ===")
print()
with open('binary_to_bcd/binary_to_bcd_tb.v', 'r') as f:
    tb_code = f.read()
    print(tb_code)

print("\n" + "="*60)
print("HOW THE TESTBENCH WORKS:")
print("="*60)
print("""
1. Instantiates the binary_to_bcd_converter module
2. Loops through all 32 possible 5-bit inputs (0 to 31)
3. For each input:
   - Applies the binary value
   - Waits 10 time units for combinational logic to settle
   - Extracts ones digit (bcd_output[3:0])
   - Extracts tens digit (bcd_output[7:4])
   - Calculates expected values using modulo and division
   - Compares actual vs expected
   - Reports error if mismatch
4. If all tests pass, displays "All test cases passed!"

Invocation: iverilog -g2012 -o sim.vvp design.v binary_to_bcd_tb.v && vvp sim.vvp
""")

=== TESTBENCH CODE ===

`timescale 1ns / 1ps

module tb_binary_to_bcd_converter;

reg [4:0] binary_input;
wire [7:0] bcd_output;

binary_to_bcd_converter uut (
    .binary_input(binary_input),
    .bcd_output(bcd_output)
);

integer i;
reg [4:0] test_binary;
reg [7:0] expected_bcd;

initial begin
    $display("Testing Binary-to-BCD Converter...");

    for (i = 0; i < 32; i++) begin
        test_binary = i;
        binary_input = test_binary;

        // Calculate expected BCD output
        expected_bcd[3:0] = test_binary % 10;
        expected_bcd[7:4] = test_binary / 10;

        #10; // Wait for the results

        if (bcd_output !== expected_bcd) begin
            $display("Error: Test case %0d failed. Expected BCD: 8'b%0b, Got: 8'b%0b",
                     test_binary, expected_bcd, bcd_output);
            $finish;
        end
    end

    $display("All test cases passed!");
    $finish;
end

reg vcd_clk;
initial begin
    $dumpfile("my_design.vcd");
    $dumpvars(0, tb_binary

## Module Template

This is the exact module interface that AutoChip must generate to match the testbench.

In [6]:
#@title Module Template

module_template = """
module binary_to_bcd_converter(
    input  [4:0] binary_input,
    output [7:0] bcd_output
);
    // Implementation here:
    // - bcd_output[3:0] = binary_input % 10  (ones digit)
    // - bcd_output[7:4] = binary_input / 10  (tens digit)
endmodule
"""

print("=== MODULE TEMPLATE ===")
print(module_template)
print("\nPort Descriptions:")
print("  • binary_input[4:0]: 5-bit binary number (range 0-31)")
print("  • bcd_output[7:0]: 8-bit BCD encoded output")
print("    - bcd_output[3:0]: ones digit (0-9)")
print("    - bcd_output[7:4]: tens digit (0-3)")
print("\nExample:")
print("  Input:  5'b10111 (23 decimal)")
print("  Output: 8'b00100011 (0x23, BCD for 23)")
print("          [7:4]=0010 (2), [3:0]=0011 (3)")

=== MODULE TEMPLATE ===

module binary_to_bcd_converter(
    input  [4:0] binary_input,
    output [7:0] bcd_output
);
    // Implementation here:
    // - bcd_output[3:0] = binary_input % 10  (ones digit)
    // - bcd_output[7:4] = binary_input / 10  (tens digit)
endmodule


Port Descriptions:
  • binary_input[4:0]: 5-bit binary number (range 0-31)
  • bcd_output[7:0]: 8-bit BCD encoded output
    - bcd_output[3:0]: ones digit (0-9)
    - bcd_output[7:4]: tens digit (0-3)

Example:
  Input:  5'b10111 (23 decimal)
  Output: 8'b00100011 (0x23, BCD for 23)
          [7:4]=0010 (2), [3:0]=0011 (3)


In [7]:
#@title Configuration

# Configuration for AutoChip
config = {
    "model_type": "openai",
    "model_id": "gpt-4o-mini",
    "max_iterations": 5,
    "num_candidates": 5,
    "work_dir": "binary_to_bcd",
    "testbench": "binary_to_bcd_tb.v",
    "module_name": "binary_to_bcd_converter"
}

# Save config
with open('binary_to_bcd/config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("Configuration:")
print(json.dumps(config, indent=2))

Configuration:
{
  "model_type": "openai",
  "model_id": "gpt-4o-mini",
  "max_iterations": 5,
  "num_candidates": 5,
  "work_dir": "binary_to_bcd",
  "testbench": "binary_to_bcd_tb.v",
  "module_name": "binary_to_bcd_converter"
}


In [8]:
#@title Display Full Configuration

print("="*60)
print("FULL CONFIGURATION (config.json)")
print("="*60)
print(json.dumps(config, indent=2))
print("\n" + "="*60)
print("PARAMETER EXPLANATIONS:")
print("="*60)
print("""
• model_type: 'openai' - Use OpenAI API
• model_id: 'gpt-4o-mini' - Specific model version
• max_iterations: 5 - Maximum refinement cycles
• num_candidates: 5 - Designs generated per iteration
• work_dir: 'binary_to_bcd' - Working directory
• testbench: 'binary_to_bcd_tb.v' - Testbench filename
• module_name: 'binary_to_bcd_converter' - Module identifier

Temperature: 0.7 (for multiple candidates) or 0.0 (single)
Max Tokens: 1024 per generation
""")

# Also save to file for evidence
with open(f"{config['work_dir']}/config.json", 'w') as f:
    json.dump(config, f, indent=2)
print(f"\n✅ Configuration saved to {config['work_dir']}/config.json")

FULL CONFIGURATION (config.json)
{
  "model_type": "openai",
  "model_id": "gpt-4o-mini",
  "max_iterations": 5,
  "num_candidates": 5,
  "work_dir": "binary_to_bcd",
  "testbench": "binary_to_bcd_tb.v",
  "module_name": "binary_to_bcd_converter"
}

PARAMETER EXPLANATIONS:

• model_type: 'openai' - Use OpenAI API
• model_id: 'gpt-4o-mini' - Specific model version
• max_iterations: 5 - Maximum refinement cycles
• num_candidates: 5 - Designs generated per iteration
• work_dir: 'binary_to_bcd' - Working directory
• testbench: 'binary_to_bcd_tb.v' - Testbench filename
• module_name: 'binary_to_bcd_converter' - Module identifier

Temperature: 0.7 (for multiple candidates) or 0.0 (single)
Max Tokens: 1024 per generation


✅ Configuration saved to binary_to_bcd/config.json


## Design Prompt and Module Template

This is the initial specification given to the LLM.

In [9]:
#@title Design Prompt (Optimized for Fast Success)

design_prompt = """
You are an expert Verilog designer. Generate ONLY valid, synthesizable Verilog code.

STRICT REQUIREMENTS:
1. Output ONLY the module code between 'module' and 'endmodule'
2. No explanations, no comments, no markdown formatting
3. Module name MUST be exactly: binary_to_bcd_converter
4. Port names and widths MUST match exactly:
   - input  [4:0] binary_input
   - output [7:0] bcd_output

FUNCTIONALITY:
Convert 5-bit binary input (range 0-31) to 8-bit BCD output.
- bcd_output[3:0] = ones digit = binary_input % 10
- bcd_output[7:4] = tens digit = binary_input / 10

IMPLEMENTATION CONSTRAINTS:
- Use ONLY combinational logic (assign statements or always @*)
- Use Verilog's division (/) and modulo (%) operators
- No sequential logic (no clocks, no registers)
- No delays (#), no initial blocks, no always_ff
- Must compile with: iverilog -g2012

EXAMPLE BEHAVIOR:
Input: 5'b00000 (0)  → Output: 8'b00000000 (BCD: 00)
Input: 5'b01010 (10) → Output: 8'b00010000 (BCD: 10)
Input: 5'b10111 (23) → Output: 8'b00100011 (BCD: 23)
Input: 5'b11111 (31) → Output: 8'b00110001 (BCD: 31)

CRITICAL: The testbench expects:
- Exact module name: binary_to_bcd_converter
- Exact port names: binary_input, bcd_output
- Combinational output (no clock dependency)
- All 32 inputs (0-31) must produce correct BCD

Generate the complete module now:
"""

print("=== OPTIMIZED DESIGN PROMPT ===")
print(design_prompt)
print("\n" + "="*60)
print("WHY THIS PROMPT IS EFFECTIVE:")
print("="*60)
print("""
1. Clear role establishment ("expert Verilog designer")
2. Explicit output format ("ONLY the module code")
3. Exact naming requirements emphasized with MUST
4. Mathematical specification with operators
5. Multiple concrete examples showing pattern
6. Explicit constraints (no delays, no initial, etc.)
7. Testbench expectations stated clearly
8. Simple enough that direct implementation is obvious

Expected Success: 1-2 iterations (very likely iteration 1)
""")

=== OPTIMIZED DESIGN PROMPT ===

You are an expert Verilog designer. Generate ONLY valid, synthesizable Verilog code.

STRICT REQUIREMENTS:
1. Output ONLY the module code between 'module' and 'endmodule'
2. No explanations, no comments, no markdown formatting
3. Module name MUST be exactly: binary_to_bcd_converter
4. Port names and widths MUST match exactly:
   - input  [4:0] binary_input
   - output [7:0] bcd_output

FUNCTIONALITY:
Convert 5-bit binary input (range 0-31) to 8-bit BCD output.
- bcd_output[3:0] = ones digit = binary_input % 10
- bcd_output[7:4] = tens digit = binary_input / 10

IMPLEMENTATION CONSTRAINTS:
- Use ONLY combinational logic (assign statements or always @*)
- Use Verilog's division (/) and modulo (%) operators
- No sequential logic (no clocks, no registers)
- No delays (#), no initial blocks, no always_ff
- Must compile with: iverilog -g2012

EXAMPLE BEHAVIOR:
Input: 5'b00000 (0)  → Output: 8'b00000000 (BCD: 00)
Input: 5'b01010 (10) → Output: 8'b00010000 (BCD

## Running AutoChip Loop

This cell executes the main AutoChip generation loop with iterative refinement.

In [10]:
#@title Run AutoChip Loop

start_time = time()

# Initialize model
model = OpenAIModel(model_id=config["model_id"])

# Run AutoChip loop
success, final_code, trajectory = verilog_loop(
    design_prompt=design_prompt,
    module_name=config["module_name"],
    testbench_file=config["testbench"],
    max_iterations=config["max_iterations"],
    model=model,
    work_dir=config["work_dir"],
    num_candidates=config["num_candidates"]
)

end_time = time()

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
print(f"\nSuccess: {success}")
print(f"Time taken: {end_time - start_time:.2f} seconds")
print(f"Total iterations: {len(trajectory)}")

if success:
    print("\n✅ Successfully generated correct RTL!")
    print("\nFinal Verilog Code:")
    print(final_code)

    # Save final design
    with open(f"{config['work_dir']}/final_design.v", 'w') as f:
        f.write(final_code)
else:
    print("\n❌ Failed to generate correct RTL within iteration limit")

# Save trajectory
with open(f"{config['work_dir']}/trajectory.json", 'w') as f:
    json.dump(trajectory, f, indent=2)

print(f"\n📝 Trajectory saved to {config['work_dir']}/trajectory.json")


ITERATION 1/5


Testing candidate 1/5...
  ❌ Test failed

Testing candidate 2/5...
  ❌ Test failed

Testing candidate 3/5...
  ❌ Test failed

Testing candidate 4/5...
  ❌ Test failed

Testing candidate 5/5...
  ✅ TEST PASSED!

FINAL RESULTS

Success: True
Time taken: 8.34 seconds
Total iterations: 1

✅ Successfully generated correct RTL!

Final Verilog Code:
module binary_to_bcd_converter(
    input  [4:0] binary_input,
    output [7:0] bcd_output
);
assign bcd_output[3:0] = binary_input % 10;
assign bcd_output[7:4] = binary_input / 10;
endmodule

📝 Trajectory saved to binary_to_bcd/trajectory.json


## Trajectory Analysis

Let's examine the generation trajectory to understand how AutoChip arrived at the solution.

In [11]:
#@title Analyze Trajectory

print("="*60)
print("TRAJECTORY ANALYSIS")
print("="*60)

for step in trajectory:
    print(f"\nIteration {step['iteration']}:")
    if step['status'] == 'success':
        print(f"  ✅ Success on candidate {step['candidate']}")
        print(f"  Output: {step['output'][:200]}...")
    else:
        print(f"  ❌ Failed")
        if 'error' in step:
            print(f"  Error: {step['error'][:200]}...")

print("\n" + "="*60)
print("Key Insights:")
print("="*60)
print(f"- Total iterations needed: {len(trajectory)}")
print(f"- Success achieved: {success}")
if success:
    success_step = [s for s in trajectory if s['status'] == 'success'][0]
    print(f"- Successful on iteration {success_step['iteration']}, candidate {success_step['candidate']}")

TRAJECTORY ANALYSIS

Iteration 1:
  ✅ Success on candidate 5
  Output: Testing Binary-to-BCD Converter...
VCD info: dumpfile my_design.vcd opened for output.
All test cases passed!
...

Key Insights:
- Total iterations needed: 1
- Success achieved: True
- Successful on iteration 1, candidate 5


## Final Verification

Run one final verification of the generated design.

In [12]:
#@title Final Verification with Explicit Command

print("="*60)
print("EXACT SIMULATION COMMANDS")
print("="*60)
print(f"\nWorking Directory: {config['work_dir']}")
print(f"\nCompilation Command:")
print(f"  iverilog -g2012 -o sim.vvp final_design.v {config['testbench']}")
print(f"\nSimulation Command:")
print(f"  vvp sim.vvp")
print(f"\nCombined (one-liner):")
print(f"  cd {config['work_dir']} && iverilog -g2012 -o sim.vvp final_design.v {config['testbench']} && vvp sim.vvp")
print("\n" + "="*60)

if success:
    print("\nRunning final verification...\n")
    success_verify, output_verify = run_iverilog_test(
        "final_design.v",
        config["testbench"],
        config["work_dir"]
    )

    print("="*60)
    print("SIMULATION OUTPUT:")
    print("="*60)
    print(output_verify)
    print("="*60)

    if success_verify:
        print("\n✅ VERIFICATION PASSED - All test cases passed!")
    else:
        print("\n❌ VERIFICATION FAILED")
else:
    print("\n⚠️  No successful design to verify")

EXACT SIMULATION COMMANDS

Working Directory: binary_to_bcd

Compilation Command:
  iverilog -g2012 -o sim.vvp final_design.v binary_to_bcd_tb.v

Simulation Command:
  vvp sim.vvp

Combined (one-liner):
  cd binary_to_bcd && iverilog -g2012 -o sim.vvp final_design.v binary_to_bcd_tb.v && vvp sim.vvp


Running final verification...

SIMULATION OUTPUT:
Testing Binary-to-BCD Converter...
VCD info: dumpfile my_design.vcd opened for output.
All test cases passed!


✅ VERIFICATION PASSED - All test cases passed!


## Summary

### What AutoChip Did:
1. **Initial Generation**: Created multiple candidate designs based on the specification
2. **Testing**: Compiled and simulated each candidate against the testbench
3. **Feedback Loop**: When tests failed, provided error messages to the LLM for refinement
4. **Iteration**: Repeated until a correct design was found or max iterations reached

### Key Parameters:
- Model: gpt-4o-mini
- Max iterations: 5
- Candidates per iteration: 5
- Temperature: 0.7 (for diversity when generating multiple candidates)

### Design Characteristics:
- Type: Combinational logic
- Complexity: Low (simple arithmetic operations)
- Challenge: Proper BCD encoding and port matching